# खर्चाचा दावा विश्लेषण

हा नोटबुक स्थानिक पावत्या प्रतिमांमधून प्रवास खर्च प्रक्रिया करण्यासाठी प्लगइन्स वापरणाऱ्या एजंट्स कसे तयार करायचे हे दर्शवितो, खर्चाचा दावा ईमेल निर्माण करतो, आणि खर्च डेटा पाय चार्ट वापरून दृष्टीकोन प्रदान करतो. एजंट कार्याच्या संदर्भावर आधारित कार्ये गतिमानपणे निवडतात.

पायऱ्या:
1. OCR एजंट स्थानिक पावती प्रतिमा प्रक्रिया करतो आणि प्रवासाचा खर्च डेटा काढतो.
2. ईमेल एजंट खर्चाचा दावा ईमेल तयार करतो.

### प्रवास खर्चाचा उदाहरणात्मक परिस्थिती:
समजा तुम्ही एका व्यवसायिक बैठकीसाठी दुसऱ्या शहरात प्रवास करत आहात. तुमच्या कंपनीकडे सर्व योग्य प्रवास-संबंधित खर्चांची परतफेड करण्याची धोरण आहे. प्रवासाचा संभाव्य खर्चांमध्ये हा विभाग आहे:
- परिवहन:
घरच्या शहरातून गंतव्य शहरापर्यंतचा परत फिरणारा एअरफेअर.
टॅक्सी किंवा राइड-हेइलिंग सेवा हवाई अड्ड्यावर जाण्यासाठी आणि येण्यासाठी.
गंतव्य शहरातील स्थानिक प्रवास (जसे सार्वजनिक वाहतूक, भाड्याने कार, किंवा टॅक्सी).

- निवास:
बैठकीच्या ठिकाणी जवळच्या मध्यम श्रेणीच्या व्यवसायिक हॉटेलमध्ये तीन रात्री राहणे.

- जेवण:
दररोज न्याहारी, जेवण, आणि रात्रीचे जेवणासाठी भोजन भत्ता, कंपनीच्या पर डियम धोरणावर आधारित.

- विविध खर्च:
हवाई अड्ड्यावरील पार्किंग शुल्क.
हॉटेलमधील इंटरनेट प्रवेश शुल्क.
टिप्स किंवा लहान सेवा शुल्क.

- कागदपत्रे:
तुम्ही सर्व पावत्या (फ्लाइट्स, टॅक्सी, हॉटेल, जेवण इ.) आणि खर्च रिपोर्ट परतफेडीसाठी सादर करता.


## आवश्यक लायब्ररी आयात करा

नोटबुकसाठी आवश्यक लायब्ररी आणि मॉड्यूल्स आयात करा.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## खर्च मॉडेल्स परिभाषित करा

 वैयक्तिक खर्चांसाठी एक Pydantic मॉडेल तयार करा आणि वापरकर्त्याच्या प्रश्नाला संरचित खर्च डेटा मध्ये रूपांतर करण्यासाठी एक ExpenseFormatter वर्ग तयार करा.

 प्रत्येक खर्च खालील स्वरूपात दर्शविला जाईल:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## साधने परिभाषित करणे - ईमेल तयार करणे

खर्च दावे सबमिट करण्यासाठी ईमेल तयार करण्यासाठी साधन फंक्शन तयार करा.
- हे साधन Microsoft Agent Framework मधील `@tool` डेकोरेटर वापरते.
- हा खर्चांची एकूण रक्कम मोजतो आणि तपशील ईमेल बॉडीमध्ये स्वरूपित करतो.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# रसीद प्रतिमांमधून प्रवास खर्च काढण्यासाठी साधन

रसीद प्रतिमांमधून प्रवास खर्च काढण्यासाठी एक साधन फंक्शन तयार करा.
- हे साधन Microsoft Agent Framework मधील `@tool` डेकोरेटर वापरते.
- हे रसीद प्रतिमा वाचते, ते base64 मध्ये एनकोड करते, आणि एजेंटसाठी विश्लेषण करण्यासाठी डेटा URI परत करते.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## खर्च प्रक्रिया करणे

एजंट्स परिभाषित करा आणि त्यांना `WorkflowBuilder` वापरून अनुक्रमिक वर्कफ्लो मध्ये वायर करा.
- OCR एजंट रसीद प्रतिमेमधून रचना केलेला खर्च डेटा काढतो `load_receipt_image` टूल वापरून.
- ईमेल एजंट काढलेला डेटा घेतो आणि `generate_expense_email` टूल वापरून व्यावसायिक खर्च दावा ईमेल तयार करतो.
- `WorkflowBuilder` आणि `add_edge` वापरून एक अनुक्रमिक पाइपलाइन तयार होते: OCR एजंट → ईमेल एजंट.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## मुख्य function

क्रमिक कार्यप्रवाह तयार करा आणि ते चालवा जेणेकरून पावतीची प्रतिमा प्रक्रिया होईल आणि खर्च दावे अ‍ॅमेल तयार केला जाईल.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**कृपापत्र**:  
हा दस्तऐवज AI भाषांतर सेवेसह [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून भाषांतरित केला आहे. आम्ही अचूकतेसाठी प्रयत्न करतो असले तरी, कृत्रिम भाषांतरांमध्ये चुका किंवा अचूकतेची कमतरता असू शकते याची कृपया नोंद घ्या. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्त्रोत मानले जावे. महत्त्वाच्या माहितीसाठी व्यावसायीक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे झालेल्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थलागी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
